# Mega Project 3 — Risk Segmentation
## Problem 5: Cross-Axis Risk-Return Synthesis — No New Modeling

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Problems 1-4 each built one real, independent segmentation axis (risk
level, external bureau behavior, instalment-loan repayment conduct,
revolving-credit usage). A portfolio or collections team's next real
question is: which of those axes, set against real regulatory capital
consumption, actually differentiates risk the most sharply? This notebook
answers that honestly, with no new modeling of any kind.

### This notebook trains no model and reads no raw Home Credit CSV
It is a pure synthesis of already-computed real outputs: Mega Project 3 /
Notebook 01's real PD/TARGET/RISK_TIER (hard dependency), Mega Project 2 /
Notebook 01's real per-applicant capital output (soft dependency -- the
"return" side of "risk-return"), and this Mega Project's own Notebooks
02-04 real segment assignments (each an independent soft dependency).
Nothing is re-scored, re-clustered, or re-fit.

### What "risk-return" means here
Not investment return -- real regulatory CAPITAL CONSUMPTION (Mega
Project 2's real Vasicek-based capital requirement) set against real
default RISK (real PD and TARGET) for each real segmentation axis this
Mega Project has built. The deliverable is a real, computed answer to
"which axis, and which segment within it, concentrates the most real risk
per unit of real capital already being held against it."

### Hard and soft dependencies
Hard dependency: Mega Project 3 / Notebook 01's real per-applicant output.
Soft dependencies (each independently optional): Mega Project 2 /
Notebook 01's real capital output, and this Mega Project's own Notebooks
02, 03, and 04's real segment outputs -- this notebook still produces a
complete, honest synthesis across whichever axes are actually present.

### Advanced error tackling applied (see LESSONS_LEARNED.md for the
### incidents each of these prevents a repeat of)
- HARD dependency on Notebook 01's real output, checked by actual required
  columns present, not just file existence.
- Four independent SOFT dependencies -- a complete, real synthesis is
  produced across whichever axes are actually available, never fabricating
  a missing one.
- `monotonic_within_noise()` applied ONLY where meaningful: Risk Tier is
  the one genuinely ORDERED axis (by real PD) -- this notebook tests
  whether real CAPITAL rate also rises monotonically through that order, a
  new real question distinct from Notebook 01's own default-rate
  monotonicity check. The other 3 axes are unordered categorical segments
  -- no monotonicity check applies, the same reasoning already established
  three times in this suite.
- No chi-square/silhouette/bootstrap section in this notebook, by design:
  those statistical-robustness questions were already asked and honestly
  answered inside each axis's own notebook (Problems 1-4) -- repeating
  them here would be double-counting, not new evidence.
- No `matplotlib.use(...)` call anywhere in this file.
- No raw Home Credit CSV is read in this notebook at all.
- No EDA section, per standing instruction.

### Verification status
Verified end-to-end on this suite's synthetic fixture via real Jupyter
execution -- 0 errors, all integrity and synthesis-validation checks pass,
HTML dashboard confirmed under a network-blocked Playwright check, Excel
workbook confirmed via LibreOffice headless recalculation. **Not yet run
against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 05 — MEGA PROJECT 3: RISK SEGMENTATION
# PROBLEM 5: CROSS-AXIS RISK-RETURN SYNTHESIS
# No New Modeling -- Reuses Mega Project 1's Real PD, Mega Project 2's Real
# Regulatory Capital, and Every Real Segmentation Axis Built in Problems 1-4
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains NO model, real or
# supervised or unsupervised, and reads NO raw Home Credit CSV at all. It is
# a pure, honest SYNTHESIS of already-computed real outputs: Mega Project 3 /
# Notebook 01's real PD/TARGET/RISK_TIER (hard dependency), Mega Project 2 /
# Notebook 01's real per-applicant EXPECTED_LOSS/CAPITAL_REQUIREMENT/
# EAD_PROXY (soft dependency), and this Mega Project's own Notebook 02
# (Bureau Segment), Notebook 03 (Repayment Segment), and Notebook 04
# (Utilization Segment) real segment assignments (each a soft dependency).
# Nothing here is re-scored, re-clustered, or re-fit -- every number in this
# notebook's output is a real join and a real aggregation of numbers other
# notebooks in this suite already computed and verified.
#
# WHAT "RISK-RETURN" MEANS HERE: not investment return -- real regulatory
# CAPITAL CONSUMPTION (Mega Project 2's real Vasicek-based capital
# requirement) set against real default RISK (Mega Project 1/3's real PD and
# TARGET) for each of the up to 4 real segmentation axes this Mega Project
# has built. A collections or portfolio team reads this as: which axis, and
# which segment within it, concentrates the most real risk per unit of real
# capital already being held against it.
#
# LESSONS APPLIED FROM THIS SUITE'S OWN HARDENING HISTORY (LESSONS_LEARNED.md
# -- every item below cites which real incident it prevents a repeat of):
#   1. HARD DEPENDENCY on Mega Project 3 / Notebook 01's real per-applicant
#      output (PD, TARGET, RISK_TIER), checked by actual required columns
#      present, not just file existence (LESSONS_LEARNED.md #4).
#   2. FOUR SOFT DEPENDENCIES (MP2 Notebook 01's capital, and this project's
#      own Notebooks 02/03/04's segments) -- each independently optional;
#      this notebook still produces a complete, real, honest synthesis
#      across whichever axes are actually present, never fabricating a
#      missing one.
#   3. `monotonic_within_noise()` APPLIED ONLY WHERE IT IS MEANINGFUL: Risk
#      Tier is the one axis in this synthesis that is genuinely ORDERED (by
#      real PD, established in Notebook 01) -- so this notebook tests
#      whether real capital rate also rises monotonically through that real
#      order (a new, real, meaningful check -- Notebook 01 already validated
#      default-rate monotonicity; this notebook validates CAPITAL-rate
#      monotonicity, which is not the same claim). The other 3 axes are
#      unordered categorical segments -- no monotonicity check applies to
#      them, the same reasoning already established three times in this
#      suite (MP2 Notebook 05, MP3 Notebooks 02-04).
#   4. NO CHI-SQUARE / SILHOUETTE / BOOTSTRAP SECTION IN THIS NOTEBOOK, BY
#      DESIGN: those statistical-robustness questions were already asked and
#      honestly answered inside each axis's OWN notebook (Problems 1-4).
#      Re-running them here on the same real segment assignments would not
#      be new evidence -- it would be double-counting. This notebook's own
#      validation (#3 above) is deliberately a DIFFERENT, new real question:
#      does real capital allocation track real risk the way it should.
#   5. No `matplotlib.use(...)` call anywhere in this file (LESSONS_LEARNED.md
#      #7).
#   6. NO RAW HOME CREDIT CSV IS READ IN THIS NOTEBOOK -- disclosed above and
#      structurally true: Section 4 loads only already-produced real
#      artifacts from this suite's own decision_engine/ folders.
#   7. NO EDA SECTION -- per standing instruction.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

SEED = int(CONFIG.get("random_seed", 42))

MP2_ARTIFACTS_DIR = SUITE_ROOT / "02_mega_project_2_regulatory_capital" / "decision_engine" / "artifacts"
MP3_DIR = SUITE_ROOT / "03_mega_project_3_risk_segmentation"
ARTIFACTS_DIR = MP3_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP3_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity
from utils.stats_checks import monotonic_within_noise

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import)
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2)
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — HARD + SOFT DEPENDENCIES. Only already-produced real artifacts
# from this suite's own decision_engine/ folders are read -- no raw Home
# Credit CSV at all (LESSON #6).
# ---------------------------------------------------------------------------
NB01_PATH = ARTIFACTS_DIR / "notebook_01_risk_tiers.csv"
if not NB01_PATH.exists():
    raise FileNotFoundError(
        "Mega Project 3 / Notebook 05 requires Mega Project 3 / Notebook 01's real "
        "per-applicant output, which has not been produced on this machine yet. Fix: run "
        "03_mega_project_3_risk_segmentation/notebooks/01_data_driven_risk_tier_construction.ipynb "
        "end-to-end first, then re-run this notebook."
    )
nb01 = pl.read_csv(NB01_PATH)
_req = ["SK_ID_CURR", "PD", "TARGET", "RISK_TIER"]
_missing = [c for c in _req if c not in nb01.columns]
if _missing:
    raise KeyError(f"Required columns missing from Notebook 01's output: {_missing}. Re-run Notebook 01.")
N_SCOPE = nb01.height
df = nb01.to_pandas()
print(f"[LOAD] Real per-applicant output from Notebook 01: {N_SCOPE:,} rows.")

# SOFT dependency: MP2 Notebook 01's real capital output (the "return" side
# of "risk-return"). Same required-columns pattern already used by MP3
# Notebook 01's own capital enrichment.
MP2_SCORES_PATH = MP2_ARTIFACTS_DIR / "notebook_01_capital_scores.csv"
CAPITAL_AVAILABLE = MP2_SCORES_PATH.exists()
if CAPITAL_AVAILABLE:
    mp2_scores = pl.read_csv(MP2_SCORES_PATH)
    _cap_req = ["SK_ID_CURR", "EXPECTED_LOSS", "CAPITAL_REQUIREMENT", "EAD_PROXY"]
    _cap_missing = [c for c in _cap_req if c not in mp2_scores.columns]
    if _cap_missing:
        print(f"[SOFT-DEPENDENCY] MP2 Notebook 01's output is missing required columns "
              f"{_cap_missing} -- skipping the real capital enrichment.")
        CAPITAL_AVAILABLE = False
    else:
        df = df.merge(mp2_scores.select(_cap_req).to_pandas(), on="SK_ID_CURR", how="left")
        print(f"[SOFT-DEPENDENCY] Real MP2 Notebook 01 capital output found -- "
              f"{mp2_scores.height:,} rows joined for the real capital ('return') side of this synthesis.")
else:
    print("[SOFT-DEPENDENCY] Real MP2 Notebook 01 capital output not found -- this notebook still "
          "produces a complete real synthesis of RISK across axes; only the capital ('return') side "
          "is skipped. Run Mega Project 2 / Notebook 01 first for the full risk-return view.")

# SOFT dependencies: this Mega Project's own Notebooks 02-04 real segments.
_soft_segment_specs = [
    ("notebook_02_bureau_segments.csv", "BUREAU_SEGMENT", "Notebook 02 (Bureau Segment)"),
    ("notebook_03_repayment_segments.csv", "REPAYMENT_SEGMENT", "Notebook 03 (Repayment Segment)"),
    ("notebook_04_utilization_segments.csv", "UTILIZATION_SEGMENT", "Notebook 04 (Utilization Segment)"),
]
AXES = [("RISK_TIER", "Risk Tier", True)]  # (column, display name, is_ordered)
for fname, col, label in _soft_segment_specs:
    path = ARTIFACTS_DIR / fname
    available = path.exists()
    if available:
        seg = pl.read_csv(path).select(["SK_ID_CURR", col])
        df = df.merge(seg.to_pandas(), on="SK_ID_CURR", how="left")
        AXES.append((col, label.split(" (")[1].rstrip(")"), False))
        print(f"[SOFT-DEPENDENCY] Real {label} output found -- included as a real cross-axis "
              f"segmentation in this synthesis.")
    else:
        print(f"[SOFT-DEPENDENCY] Real {label} output not found -- this notebook still produces a "
              f"complete real synthesis; that one axis is skipped.")

print(f"[SYNTHESIS] {len(AXES)} real segmentation axis(es) available for this synthesis: "
      f"{', '.join(name for _, name, _ in AXES)}.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real per-axis segment aggregation (no new modeling -- pure,
# honest aggregation of already-computed real numbers).
# ---------------------------------------------------------------------------
def real_axis_aggregation(frame: pd.DataFrame, seg_col: str) -> pd.DataFrame:
    """Real, vectorized per-segment aggregation for one real segmentation
    axis: real population, real default rate, real mean PD, and (only when
    the real capital columns are present on `frame`) real total capital
    requirement, real total EAD, real capital-rate-of-EAD, and real total
    expected loss. No modeling -- pure pandas aggregation."""
    agg_spec = {"n_applicants": ("SK_ID_CURR", "size"), "real_default_rate": ("TARGET", "mean"),
                "mean_pd": ("PD", "mean")}
    if CAPITAL_AVAILABLE:
        agg_spec.update({
            "total_capital_requirement": ("CAPITAL_REQUIREMENT", "sum"),
            "total_ead": ("EAD_PROXY", "sum"),
            "total_expected_loss": ("EXPECTED_LOSS", "sum"),
        })
    out = frame.groupby(seg_col, observed=True).agg(**agg_spec).reset_index()
    out["n_applicants"] = out["n_applicants"].astype(int)
    if CAPITAL_AVAILABLE:
        out["capital_rate_of_ead"] = out["total_capital_requirement"] / out["total_ead"].replace(0, np.nan)
        out["el_rate_of_ead"] = out["total_expected_loss"] / out["total_ead"].replace(0, np.nan)
    return out


axis_aggregations: dict[str, pd.DataFrame] = {}
for col, name, is_ordered in AXES:
    scoped = df[df[col].notna()] if col != "RISK_TIER" else df
    agg = real_axis_aggregation(scoped, col)
    if col == "RISK_TIER":
        agg[col] = pd.Categorical(agg[col], categories=[f"Tier {i + 1}" for i in range(agg.shape[0])], ordered=True)
        agg = agg.sort_values(col).reset_index(drop=True)
    axis_aggregations[col] = agg
    _cap_note = (f", real capital rate {agg['capital_rate_of_ead'].min():.2%}-{agg['capital_rate_of_ead'].max():.2%}"
                 if CAPITAL_AVAILABLE else "")
    print(f"[AXIS] {name}: {agg.shape[0]} real segments, real default rate "
          f"{agg['real_default_rate'].min():.2%}-{agg['real_default_rate'].max():.2%}{_cap_note}.")

# ---------------------------------------------------------------------------
# SECTION 6 — Real cross-axis comparison: which axis differentiates real
# risk (and, when available, real capital) most sharply? This is the actual
# SYNTHESIS deliverable -- descriptive, not gated pass/fail.
# ---------------------------------------------------------------------------
cross_axis_rows = []
for col, name, _ in AXES:
    agg = axis_aggregations[col]
    row = {
        "axis": name, "n_segments": int(agg.shape[0]),
        "default_rate_min": float(agg["real_default_rate"].min()),
        "default_rate_max": float(agg["real_default_rate"].max()),
        "default_rate_spread": float(agg["real_default_rate"].max() - agg["real_default_rate"].min()),
    }
    if CAPITAL_AVAILABLE:
        row.update({
            "capital_rate_min": float(agg["capital_rate_of_ead"].min()),
            "capital_rate_max": float(agg["capital_rate_of_ead"].max()),
            "capital_rate_spread": float(agg["capital_rate_of_ead"].max() - agg["capital_rate_of_ead"].min()),
        })
    cross_axis_rows.append(row)
cross_axis_summary = pd.DataFrame(cross_axis_rows).sort_values("default_rate_spread", ascending=False).reset_index(drop=True)
_widest = cross_axis_summary.iloc[0]
print(f"[SYNTHESIS] Real default-rate spread by axis (widest to narrowest): " +
      ", ".join(f"{r['axis']}={r['default_rate_spread']:.2%}" for _, r in cross_axis_summary.iterrows()))
print(f"[SYNTHESIS] Widest real risk differentiation: {_widest['axis']} "
      f"({_widest['default_rate_spread']:.2%} spread across {int(_widest['n_segments'])} real segments).")

# ---------------------------------------------------------------------------
# SECTION 7 — Real validation, specific to this synthesis (LESSON #3/#4):
# does real capital rate rise monotonically through Risk Tier's real,
# PD-ordered axis? This is a NEW real question -- Notebook 01 already
# validated real DEFAULT-RATE monotonicity by tier; this validates real
# CAPITAL-RATE monotonicity, a different claim.
# ---------------------------------------------------------------------------
CAPITAL_MONOTONIC_BY_TIER = None
if CAPITAL_AVAILABLE:
    tier_agg = axis_aggregations["RISK_TIER"]
    rates_highest_first = tier_agg["capital_rate_of_ead"].tolist()[::-1]
    counts_highest_first = tier_agg["n_applicants"].tolist()[::-1]
    CAPITAL_MONOTONIC_BY_TIER, _cap_mono_detail = monotonic_within_noise(rates_highest_first, counts_highest_first, alpha=0.05)
    print(f"[VALIDATION] Real capital-rate monotonicity across data-driven Risk Tiers (statistically-"
          f"tolerant Bonferroni-corrected check): {'HOLDS' if CAPITAL_MONOTONIC_BY_TIER else 'DOES NOT HOLD'}.")
else:
    print("[VALIDATION] Real capital output unavailable -- skipping the capital-rate monotonicity check "
          "(soft dependency; this notebook's risk-only synthesis is unaffected).")

SYNTHESIS_VALIDATION_CHECKS = [
    ("every_risk_tier_applicant_covered", int(axis_aggregations["RISK_TIER"]["n_applicants"].sum()) == N_SCOPE),
    ("no_negative_default_rates", bool(all((axis_aggregations[c]["real_default_rate"] >= 0).all() for c, _, _ in AXES))),
]
if CAPITAL_AVAILABLE:
    SYNTHESIS_VALIDATION_CHECKS.append(
        ("no_negative_capital_rates", bool(all((axis_aggregations[c]["capital_rate_of_ead"] >= 0).all() for c, _, _ in AXES)))
    )
    SYNTHESIS_VALIDATION_CHECKS.append(("capital_rate_monotonic_by_risk_tier", bool(CAPITAL_MONOTONIC_BY_TIER)))
SYNTHESIS_ROBUST = all(ok for _, ok in SYNTHESIS_VALIDATION_CHECKS)
_failed_synthesis_checks = [name for name, ok in SYNTHESIS_VALIDATION_CHECKS if not ok]
SYNTHESIS_VERDICT = (
    "SYNTHESIS VALIDATED — CAPITAL TRACKS RISK AS EXPECTED" if SYNTHESIS_ROBUST
    else "SYNTHESIS FLAGGED — failed: " + ", ".join(_failed_synthesis_checks) +
         " (this notebook's own equivalent of a statistical-robustness verdict -- a real, computed "
         "finding, not a code defect; see the model card if capital does not track risk as expected)"
)
for name, ok in SYNTHESIS_VALIDATION_CHECKS:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Synthesis verdict: {SYNTHESIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 8 — Inline charts. No matplotlib.use(...) call (LESSON #5).
# ---------------------------------------------------------------------------
n_panels = 3 if CAPITAL_AVAILABLE else 2
fig, axes_plt = plt.subplots(1, n_panels, figsize=(19 if n_panels == 3 else 13, 5))
tier_agg = axis_aggregations["RISK_TIER"]
axes_plt[0].bar(tier_agg["RISK_TIER"].astype(str), tier_agg["real_default_rate"], color=_palette(len(tier_agg)))
axes_plt[0].set_ylabel("Real Default Rate"); axes_plt[0].set_title("Real Default Rate by Risk Tier")
plt.setp(axes_plt[0].get_xticklabels(), rotation=30, ha="right")
axes_plt[1].bar(cross_axis_summary["axis"], cross_axis_summary["default_rate_spread"], color=_palette(len(cross_axis_summary)))
axes_plt[1].set_ylabel("Real Default Rate Spread"); axes_plt[1].set_title("Real Risk Differentiation by Axis")
plt.setp(axes_plt[1].get_xticklabels(), rotation=30, ha="right")
if CAPITAL_AVAILABLE:
    axes_plt[2].bar(tier_agg["RISK_TIER"].astype(str), tier_agg["capital_rate_of_ead"], color=_palette(len(tier_agg)))
    axes_plt[2].set_ylabel("Real Capital Rate of EAD"); axes_plt[2].set_title("Real Capital Rate by Risk Tier "
                                                                               "(MP2 enrichment)")
    plt.setp(axes_plt[2].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_05_cross_axis_synthesis.png", dpi=110)
plt.show()

# ---------------------------------------------------------------------------
# SECTION 9 — Pipeline Integrity Checks (structural)
# ---------------------------------------------------------------------------
checks = [
    ("real_data_loaded", N_SCOPE > 0),
    ("required_columns_present", len(_missing) == 0),
    ("at_least_one_axis_available", len(AXES) >= 1),
    ("every_axis_sums_to_scope", bool(all(int(axis_aggregations[c]["n_applicants"].sum()) == N_SCOPE for c, _, _ in AXES))),
    ("cross_axis_summary_matches_axis_count", cross_axis_summary.shape[0] == len(AXES)),
    ("cpu_thread_ceiling_applied_before_import", os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Pipeline integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 10 — Reporting & Packaging (SOP Stage 5)
# ---------------------------------------------------------------------------
csv_tables = {f"notebook_05_{col.lower()}_aggregation": agg for col, agg in axis_aggregations.items()}
csv_tables["notebook_05_cross_axis_summary"] = cross_axis_summary
csv_paths = write_csv_outputs(csv_tables, REPORTS_DIR)
df.to_csv(ARTIFACTS_DIR / "notebook_05_synthesis.csv", index=False)

ASSUMPTIONS = {"AXES_AVAILABLE": len(AXES), "CAPITAL_AVAILABLE": CAPITAL_AVAILABLE}
ASSUMPTION_NOTES = {
    "AXES_AVAILABLE": "Real segmentation axes actually joined into this synthesis (Risk Tier is always "
                       "present; Bureau/Repayment/Utilization Segment are each an independent soft "
                       "dependency on the corresponding earlier notebook's real output).",
    "CAPITAL_AVAILABLE": "Whether Mega Project 2 / Notebook 01's real capital output was found -- when "
                          "False, this notebook still produces a complete real risk-only synthesis; the "
                          "capital ('return') side of the risk-return view is simply omitted.",
}

STORY_SYNTHESIS = [
    f"Real default-rate spread by axis (widest to narrowest): " +
    ", ".join(f"{r['axis']}={r['default_rate_spread']:.2%}" for _, r in cross_axis_summary.iterrows()) + ".",
    f"Synthesis verdict: {SYNTHESIS_VERDICT}" if CAPITAL_AVAILABLE else
    "Real capital output unavailable -- risk-only synthesis; re-run after Mega Project 2 / Notebook 01 "
    "for the full risk-return view.",
]
INSIGHTS = [{
    "headline": f"{_widest['axis']} differentiates real risk most sharply "
                f"({_widest['default_rate_spread']:.2%} default-rate spread across {int(_widest['n_segments'])} "
                f"real segments)",
    "specific": STORY_SYNTHESIS[0],
    "measurable": f"{len(AXES)} real segmentation axis(es) synthesized across {N_SCOPE:,} real applicants" +
                  (f"; real capital enrichment included." if CAPITAL_AVAILABLE else "; real capital enrichment "
                   "not available this run."),
    "achievable": f"Computed end-to-end in {round(time.time() - T0, 1)}s from already-produced real outputs "
                  f"-- no new modeling, no raw data re-read.",
    "relevant": "Gives a collections or portfolio-management team a single, honest cross-axis view of "
                "which real segmentation (risk level, external bureau behavior, instalment repayment "
                "conduct, or revolving-credit usage) most sharply separates real risk and real capital "
                "consumption, to prioritize which axis to act on.",
    "timebound": "Re-run after any upstream notebook (Notebook 01, MP2 Notebook 01, or this project's "
                 "Notebooks 02-04) re-run refreshes its real output.",
}]

word_sections = [{
    "heading": "Real Cross-Axis Risk-Return Summary",
    "paragraphs": ["Real default-rate (and, when available, real capital-rate) spread per real "
                   "segmentation axis -- the wider the spread, the more sharply that axis differentiates "
                   "real risk."],
    "table": {"headers": list(cross_axis_summary.columns),
              "rows": cross_axis_summary.astype(object).values.tolist()},
    "image_path": ARTIFACTS_DIR / "notebook_05_cross_axis_synthesis.png", "story": STORY_SYNTHESIS,
}]
word_path = build_word_report(
    REPORTS_DIR / "notebook_05_report.docx",
    title="Mega Project 3 — Notebook 05: Cross-Axis Risk-Return Synthesis",
    subtitle=f"{len(AXES)} real segmentation axis(es), no new modeling",
    exec_summary=[
        f"{N_SCOPE:,} real applicants synthesized across {len(AXES)} real segmentation axis(es): "
        f"{', '.join(name for _, name, _ in AXES)}.",
        f"Widest real risk differentiation: {_widest['axis']} ({_widest['default_rate_spread']:.2%} spread).",
        f"Synthesis verdict: {SYNTHESIS_VERDICT}",
    ],
    sections=word_sections, insights=INSIGHTS,
)

excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_05_workbook.xlsx",
    assumptions=ASSUMPTIONS, assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Cross-Axis Summary", "headers": list(cross_axis_summary.columns),
         "rows": cross_axis_summary.astype(object).values.tolist(), "highlight_col": "default_rate_spread"},
        {"name": "Risk Tier Detail", "headers": list(axis_aggregations["RISK_TIER"].columns),
         "rows": axis_aggregations["RISK_TIER"].astype(object).values.tolist()},
    ],
    insights_sheet={"name": "SMART Insights", "items": INSIGHTS},
)

kpi_cards = [
    {"label": "Real Applicants", "value": f"{N_SCOPE:,}"},
    {"label": "Real Axes Synthesized", "value": str(len(AXES))},
    {"label": "Widest Risk Differentiation", "value": _widest["axis"]},
    {"label": "Synthesis Verdict", "value": "VALIDATED" if SYNTHESIS_ROBUST else "FLAGGED"},
]
charts = [
    {"id": "defaultRateSpreadByAxis", "title": "Real Risk Differentiation by Axis", "type": "bar",
     "labels": cross_axis_summary["axis"].tolist(),
     "datasets": [{"label": "Real Default Rate Spread", "data": cross_axis_summary["default_rate_spread"].tolist(),
                   "backgroundColor": _palette(len(cross_axis_summary))}], "story": STORY_SYNTHESIS},
    {"id": "defaultRateByTier", "title": "Real Default Rate by Risk Tier", "type": "bar",
     "labels": tier_agg["RISK_TIER"].astype(str).tolist(),
     "datasets": [{"label": "Real Default Rate", "data": tier_agg["real_default_rate"].tolist(),
                   "backgroundColor": _palette(len(tier_agg))}]},
]
html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_05_dashboard.html",
    title="Mega Project 3 — Cross-Axis Risk-Return Synthesis",
    subtitle=f"{N_SCOPE:,} real applicants — {len(AXES)} real segmentation axis(es), no new modeling",
    kpi_cards=kpi_cards, charts=charts, insights=INSIGHTS,
    data_table={"title": "Cross-Axis Summary (real)", "columns": list(cross_axis_summary.columns),
                "rows": cross_axis_summary.values.tolist(), "filter_column": "axis"},
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s).")

# ---------------------------------------------------------------------------
# SECTION 11 — Save artifacts + governance stamp (idempotent)
# ---------------------------------------------------------------------------
summary = {
    "notebook": "05_cross_axis_risk_return_synthesis",
    "mega_project": "Mega Project 3 - Risk Segmentation",
    "problem": "Problem 5 - Cross-Axis Risk-Return Synthesis",
    "random_seed": SEED,
    "n_applicants": N_SCOPE,
    "upstream_dependency": {"source_notebook": "Mega Project 3 / Notebook 01", "reused_not_recomputed": True,
                             "columns_reused": _req},
    "soft_dependencies": {
        "capital": {"source": "Mega Project 2 / Notebook 01", "available": CAPITAL_AVAILABLE},
        "bureau_segment": {"source": "Mega Project 3 / Notebook 02", "available": "BUREAU_SEGMENT" in df.columns},
        "repayment_segment": {"source": "Mega Project 3 / Notebook 03", "available": "REPAYMENT_SEGMENT" in df.columns},
        "utilization_segment": {"source": "Mega Project 3 / Notebook 04", "available": "UTILIZATION_SEGMENT" in df.columns},
    },
    "axes_synthesized": [name for _, name, _ in AXES],
    "axis_aggregations": {col: agg.to_dict(orient="records") for col, agg in axis_aggregations.items()},
    "cross_axis_summary": cross_axis_summary.to_dict(orient="records"),
    "capital_rate_monotonic_by_risk_tier": CAPITAL_MONOTONIC_BY_TIER,
    "synthesis_validation": {
        "validation_checks": {name: bool(ok) for name, ok in SYNTHESIS_VALIDATION_CHECKS},
        "failed_validation_checks": _failed_synthesis_checks,
        "verdict": SYNTHESIS_VERDICT,
        "note": "This notebook's own equivalent of a statistical-robustness verdict -- no chi-square/"
                "silhouette here by design (LESSON #4): those questions were already answered inside "
                "each axis's own notebook. This validates a NEW real question: does real capital "
                "allocation track real risk through Risk Tier's real, PD-ordered axis.",
    },
    "integrity_checks": {n: bool(ok) for n, ok in checks},
    "reporting_artifacts": [word_path.name, excel_path.name, html_path.name] + [f"{s}.csv" for s in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_05_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[DONE] Mega Project 3 / Notebook 05 complete in {summary['runtime_seconds']}s. "
      f"{len(AXES)} real segmentation axis(es) synthesized. Synthesis verdict: {SYNTHESIS_VERDICT}.")
